# Modèles pour le Produit FOODS_3_090

Dans cette étude, nous avons cherché à prévoir les ventes du produit FOODS_3_090 à partir du dataset M5. Pour cela, nous avons utilisé deux approches différentes : XGBoost et SARIMA, afin de comparer leurs performances et identifier le modèle le plus adapté.

Le choix de ces modèles repose sur la volonté de confronter deux méthodes de prévision des ventes aux approches différentes. Le modèle XGBoost a été retenu car il est particulièrement performant pour les problèmes de régression et capable de capturer des relations complexes entre plusieurs variables. Grâce aux variables explicatives créées (lags, moyennes mobiles, mois, jour de la semaine), il permet de mieux représenter les tendances et variations des ventes.
De son côté, le modèle SARIMA a été choisi car il est spécialement conçu pour l’analyse des séries temporelles. Il prend en compte la tendance, l’autocorrélation ainsi que la saisonnalité des données, notamment les cycles hebdomadaires observés dans les ventes.

L’objectif de cette démarche est donc de comparer un modèle de machine learning moderne (XGBoost) avec un modèle statistique classique de séries temporelles (SARIMA) afin d’identifier l’approche la plus efficace pour la prévision des ventes du produit FOODS_3_090.

## 1) XGBoost

Le modèle XGBoost repose sur le machine learning. Nous avons d’abord préparé les données en transformant les ventes journalières en série temporelle, puis en ajoutant plusieurs variables explicatives comme le jour de la semaine, le mois, la saison, les ventes passées (lags) et les moyennes mobiles. Ces informations permettent au modèle de mieux apprendre les tendances de vente. Après l’entraînement, les performances ont été évaluées avec les métriques MAE et RMSE.

In [6]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")


product = sales[sales["item_id"] == "FOODS_3_090"]

print(product.shape)
print(product.head())


id_columns = [
    "id", "item_id", "dept_id", "cat_id",
    "store_id", "state_id"
]

df_long = product.melt(
    id_vars=id_columns,
    var_name="d",
    value_name="sales"
)


df = df_long.merge(
    calendar[["d", "date"]],
    on="d",
    how="left"
)

df["date"] = pd.to_datetime(df["date"])


df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return 0  
    elif month in [3, 4, 5]:
        return 1  
    elif month in [6, 7, 8]:
        return 2  
    else:
        return 3  

df["season"] = df["month"].apply(get_season)


df["lag_7"] = df["sales"].shift(7)
df["lag_14"] = df["sales"].shift(14)
df["lag_28"] = df["sales"].shift(28)

df["rolling_mean_7"] = df["sales"].shift(1).rolling(7).mean()
df["rolling_mean_30"] = df["sales"].shift(1).rolling(30).mean()
df["rolling_std_30"] = df["sales"].shift(1).rolling(30).std()

df = df.dropna()


features = [
    "day_of_week",
    "month",
    "season",
    "is_weekend",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_std_30"
]

X = df[features]
y = df["sales"]

train_size = int(len(df) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]


model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("MAE :", mae)
print("RMSE :", rmse)

(10, 1919)
                                id      item_id  dept_id cat_id store_id  \
2314   FOODS_3_090_CA_1_validation  FOODS_3_090  FOODS_3  FOODS     CA_1   
5363   FOODS_3_090_CA_2_validation  FOODS_3_090  FOODS_3  FOODS     CA_2   
8412   FOODS_3_090_CA_3_validation  FOODS_3_090  FOODS_3  FOODS     CA_3   
11461  FOODS_3_090_CA_4_validation  FOODS_3_090  FOODS_3  FOODS     CA_4   
14510  FOODS_3_090_TX_1_validation  FOODS_3_090  FOODS_3  FOODS     TX_1   

      state_id  d_1  d_2  d_3  d_4  ...  d_1904  d_1905  d_1906  d_1907  \
2314        CA  107  182   47   47  ...      47      82      83      30   
5363        CA  116   90   35   33  ...      70      79      62      65   
8412        CA  108  132  102  120  ...     129     160     204      86   
11461       CA   69   49   37   40  ...      48      32      36      30   
14510       TX   75   54   74   62  ...      60      68      69      48   

       d_1908  d_1909  d_1910  d_1911  d_1912  d_1913  
2314       45      29    

Les résultats du modèle XGBoost montrent une performance correcte pour la prévision des ventes du produit FOODS_3_090.

On obtient un MAE de 18,52, ce qui signifie qu’en moyenne, les prédictions s’écartent d’environ 18 unités des ventes réelles. Le RMSE de 28,60 indique que certaines erreurs sont plus importantes, mais restent globalement maîtrisées.

Ces résultats montrent que le modèle est capable de capter les tendances générales des ventes grâce aux variables explicatives (lags, moyennes mobiles, variables temporelles). Cependant, il peut encore avoir des difficultés sur certaines variations brusques ou imprévisibles des ventes.

## 2) SARIMA 

Le modèle SARIMA est un modèle statistique de séries temporelles. Contrairement à XGBoost, il utilise directement l’historique des ventes sans créer beaucoup de variables supplémentaires. Il permet de prendre en compte la tendance, la saisonnalité et les répétitions hebdomadaires des ventes. Le modèle a ensuite été entraîné puis évalué avec les mêmes métriques afin de comparer les résultats avec XGBoost.

In [9]:
import pandas as pd
import numpy as np
import os

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error


print("Dossier actuel :")
print(os.getcwd())

print("\nFichiers présents :")
print(os.listdir())


sales = pd.read_csv("sales_train_validation.csv")
calendar = pd.read_csv("calendar.csv")



print("\nDataset chargé avec succès !")


product = sales[sales["item_id"] == "FOODS_3_090"]

print("\nNombre de lignes pour FOODS_3_090 :")
print(product.shape)

print("\nAperçu :")
print(product.head())


id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

df_long = product.melt(
    id_vars=id_columns,
    var_name="d",
    value_name="sales"
)


df = df_long.merge(
    calendar[["d", "date"]],
    on="d",
    how="left"
)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date")

print("\nDonnées avec dates :")
print(df.head())

series = df["sales"].astype(float)

print("\nLongueur de la série :", len(series))

train_size = int(len(series) * 0.8)

train = series[:train_size]
test = series[train_size:]

print("\nTrain size :", len(train))
print("Test size :", len(test))


model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

print("\nEntraînement du modèle SARIMA...")

result = model.fit(disp=False)

print("\nModèle entraîné avec succès !")


predictions = result.forecast(steps=len(test))


mae = mean_absolute_error(test, predictions)
rmse = np.sqrt(mean_squared_error(test, predictions))

print("\n===== RÉSULTATS SARIMA =====")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))


comparison = pd.DataFrame({
    "Reel": test.values,
    "Prediction": predictions.values
})

print("\nAperçu des prédictions :")
print(comparison.head(20))

Dossier actuel :
C:\Users\marie

Fichiers présents :
['.android', '.conda', '.condarc', '.continuum', '.dotnet', '.emulator_console_auth_token', '.gitconfig', '.gradle', '.ipynb_checkpoints', '.ipython', '.javacpp', '.junie', '.jupyter', '.matplotlib', '.node_repl_history', '.packettracer', '.skiko', '.spyder-py3', '.ssh', '.streamlit', '.templateengine', '.VirtualBox', '.vscode', 'anaconda3', "Analyse d'un Produit .ipynb", 'AndroidStudioProjects', 'AppData', 'Application Data', 'cache', 'calendar.csv', 'Cisco Packet Tracer 8.2.2', 'Contacts', 'Cookies', 'Documents', 'Downloads', 'Exploration Des Données.ipynb', 'Favorites', 'Jedi', 'Links', 'Local Settings', 'Menu Démarrer', 'Mes documents', 'modele_depression.pkl', 'modele_depression1.pkl', 'Modèles', 'Modèles Pour le produit FOODS_3_090.ipynb', 'Music', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{a2332f18-cdbf-11ec-8680-002248483d79}.TM.blf', 'NTUSER.DAT{a2332f18-cdbf-11ec-8680-002248483d79}.TMContainer0000000000

C:\Users\marie\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\marie\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Modèle entraîné avec succès !

===== RÉSULTATS SARIMA =====
MAE  : 30.06
RMSE : 40.17

Aperçu des prédictions :
    Reel  Prediction
0   28.0   39.668670
1   27.0   50.161599
2   22.0   48.457555
3   51.0   48.316832
4   12.0   47.189326
5   17.0   50.604813
6   29.0   45.033872
7   19.0   48.293761
8   28.0   48.317453
9   49.0   47.730619
10  75.0   48.366372
11  68.0   48.236416
12  36.0   49.129743
13  32.0   47.549778
14  89.0   47.929563
15  40.0   48.368631
16  88.0   47.737469
17  13.0   48.342417
18  19.0   48.172885
19  49.0   49.166275


C:\Users\marie\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
C:\Users\marie\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


Les résultats du modèle SARIMA montrent une performance plus faible que celle du modèle XGBoost sur la prévision des ventes du produit FOODS_3_090.

On obtient un MAE de 30,06, ce qui indique une erreur moyenne plus élevée entre les valeurs réelles et prédites. Le RMSE de 40,17 confirme également une plus grande dispersion des erreurs, avec certaines prédictions assez éloignées des valeurs réelles.

On remarque aussi que les prédictions sont relativement lissées autour d’une valeur moyenne (~48), ce qui montre que le modèle capture la tendance globale mais réagit mal aux variations réelles des ventes.

Ainsi, le modèle SARIMA est moins précis que XGBoost sur ce jeu de données, car il a plus de difficulté à suivre les fluctuations importantes des ventes.

# CONCLUSION

XGBoost obtient de meilleures performances que SARIMA avec un MAE et RMSE plus faibles. Cela montre qu’un modèle de machine learning utilisant des variables explicatives est plus adapté à ce type de données que le modèle statistique classique.